# Pass 2 — Schnelltest

Fuehrt nur Pass 2 durch (politische_stroemung, DK-Index, quote_amplification_index, manipulation_targets) — Gegenstueck zu `pass1_test.ipynb`. Pass 2 laeuft auf dem vollen, unanonymisierten Originaltext (`analyzer.py`), nutzt also `00_original_text.txt`, nicht die anonymisierte/zitatbereinigte Variante.

Wendet nach dem LLM-Call dieselbe Nachbearbeitung wie `analyzer.py` an: `normalize_role` → `_validate_manipulation_target_grounding` fuer `manipulation_targets`, `normalize_stroemung` → `_validate_stroemung_grounding` fuer `politische_stroemung` (ADR 0009/0011).

**Voraussetzung:** laufende ChromaDB (fuer `normalize_role`/`normalize_stroemung`).

## 1 — Setup

In [ ]:
import sys, json, re, os, traceback
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, "../src")

from news_analyser.prompts import load_prompt
import llm_adapter

DEBUG    = Path("../data/debug_last_run")
provider = os.environ.get("LLM_PROVIDER", "openai")
print(f"LLM_PROVIDER={provider}")

try:
    adapter = llm_adapter.get_instance(provider)
    print(f"Provider: {adapter.name}  |  Modell: {adapter.model}")
except Exception:
    traceback.print_exc()

## 2 — Artefakte laden

`00_original_text.txt` — Pass 2 sieht den vollen Originaltext, keine Anonymisierung, keine Zitat-Bereinigung.

In [ ]:
original_text = (DEBUG / "00_original_text.txt").read_text(encoding="utf-8").strip()
word_count = len(original_text.split())

# Basisdaten aus dem letzten Gesamtergebnis uebernehmen, falls vorhanden —
# Pass 2 bekommt url/domain/title/... als Kontext mit (analyzer.py: base_meta).
prev_file = DEBUG / "06_final_result.json"
if prev_file.exists():
    prev = json.loads(prev_file.read_text(encoding="utf-8"))
    base_meta = {
        "url": prev.get("source_url", "local://debug_last_run"),
        "domain": prev.get("domain", "debug"),
        "title": prev.get("title", ""),
        "author": prev.get("author", ""),
        "published_at": prev.get("published_at", ""),
        "word_count": word_count,
    }
else:
    base_meta = {
        "url": "local://debug_last_run", "domain": "debug", "title": "",
        "author": "", "published_at": "", "word_count": word_count,
    }

print(f"{word_count} Woerter")
print(base_meta)

## 3 — Pass 2 ausfuehren

In [ ]:
from news_analyser.repositories.role_store import format_roles_for_prompt

pass2_input = {**base_meta, "text": original_text}
pass2_prompt = load_prompt("system", "pass2", context={"ROLES": format_roles_for_prompt()})

print("Pass 2 laeuft ...", flush=True)
raw = adapter.generate(system_prompt=pass2_prompt, input_data=pass2_input)
print("Fertig.")

## 4 — Ergebnis parsen & nachbearbeiten

In [ ]:
from news_analyser.agents.analyzer import _validate_manipulation_target_grounding, _validate_stroemung_grounding
from news_analyser.repositories.role_store import normalize_role
from news_analyser.repositories.stroemung_store import normalize_stroemung

def extract_json(raw):
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        try:
            from json_repair import repair_json
            return json.loads(repair_json(cleaned))
        except Exception as e:
            print(f"Parse-Fehler: {e}")
            print("Raw:", raw[:500])
            return {}

result = extract_json(raw)

# Strömung: normalisieren, dann grounden (ADR 0009/0011)
stroemung = result.get("politische_stroemung", ["neutral"])
for item in stroemung:
    if isinstance(item, dict) and isinstance(item.get("label"), str):
        item["label"] = normalize_stroemung(item["label"])
n_stroemung_raw = len(stroemung)
stroemung = _validate_stroemung_grounding(stroemung, original_text)
n_stroemung_grounded = len(stroemung)

# Manipulation Targets: Rollen normalisieren, dann grounden (ADR 0007)
targets = result.get("manipulation_targets", [])
for t in targets:
    if isinstance(t, dict) and isinstance(t.get("rolle"), str):
        t["rolle"] = normalize_role(t["rolle"])
n_targets_raw = len(targets)
targets = _validate_manipulation_target_grounding(targets, original_text)
n_targets_grounded = len(targets)

print(f"DK-Index:              {result.get('dunning_kruger_index')}")
print(f"DK-Begruendung:         {result.get('dunning_kruger_explanation')}")
print(f"Quote-Amplification:    {result.get('quote_amplification_index')}")
print(f"Quote-Amp-Begruendung:  {result.get('quote_amplification_explanation')}")
print(f"Themenbereich:          {result.get('themenbereich')}")
print(f"Stroemung:              {n_stroemung_raw} roh -> {n_stroemung_grounded} nach Grounding")
print(f"Manipulation Targets:   {n_targets_raw} roh -> {n_targets_grounded} nach Grounding")

## 5 — Politische Strömung als Tabelle

In [ ]:
if not stroemung:
    print("Keine (gegroundeten) Stroemungs-Labels.")
else:
    rows = [{"Label": s.get("label") if isinstance(s, dict) else s,
             "Zitat": (s.get("quote") or "")[:90] if isinstance(s, dict) else ""}
            for s in stroemung]
    display(pd.DataFrame(rows))

## 6 — Manipulation Targets als Tabelle

In [ ]:
if not targets:
    print("Keine (gegroundeten) Manipulation Targets.")
else:
    rows = [{"Entity": t.get("entity"), "Rolle": t.get("rolle"), "Direction": t.get("direction"),
             "Rolle-Zitat": (t.get("rolle_quote") or "")[:70],
             "Direction-Zitat": (t.get("direction_quote") or "")[:70]}
            for t in targets]
    pd.set_option("display.max_colwidth", 70)
    display(pd.DataFrame(rows))

## 7 — Vergleich mit letztem Gesamtergebnis

In [ ]:
if prev_file.exists():
    prev_ft = prev.get("framing_target", {})
    cmp = pd.DataFrame([
        {"Run": "Letzter Run", "DK": prev_ft.get("dunning_kruger_index"),
         "Quote-Amp": prev_ft.get("quote_amplification_index"),
         "Stroemung": ", ".join(s.get("label") if isinstance(s, dict) else s for s in prev.get("politische_stroemung", [])),
         "Modell": prev.get("llm_model", "-")},
        {"Run": "Dieser Run", "DK": result.get("dunning_kruger_index"),
         "Quote-Amp": result.get("quote_amplification_index"),
         "Stroemung": ", ".join(s.get("label") if isinstance(s, dict) else s for s in stroemung),
         "Modell": adapter.model},
    ]).set_index("Run")
    display(cmp)
else:
    print("06_final_result.json nicht gefunden.")